# Momentum-scale map from \(K^0_S\to\pi^+\pi^-\)

This notebook builds a smooth momentum-scale correction

\[
s=s(\mathrm{side},q,p_T,\phi,\eta)
\]

from the per-daughter samples written by `ExtractK0sMomentumScaleSamples.C`.

The map-building strategy is deliberately simpler than the direct
\((\Delta r,r\Delta\phi)\) vote-map approach:

1. solve a momentum scale for every daughter using the PDG \(K^0_S\) mass and exact transverse pointing;
2. bin those scale measurements in side, charge, \(p_T\), \(\phi\), and \(\eta\);
3. use a robust median in each populated bin;
4. smoothly interpolate sparse or empty bins;
5. apply the map to independent pair candidates and compare invariant masses before and after correction.

The \(K^0_S\) sample is the calibration source. The \(\Lambda\), \(\bar\Lambda\),
\(\phi\), \(D^0\), and \(\bar D^0\) masses are independent validation channels.


In [ ]:
from pathlib import Path
import math
import numpy as np
import ROOT as root

%jsroot on

root.gStyle.SetOptStat(0)
root.gStyle.SetOptTitle(0)
root.gStyle.SetPadTickX(1)
root.gStyle.SetPadTickY(1)
root.gStyle.SetPalette(root.kBird)


## Configuration

The default binning is intentionally moderate. The raw map should first be
statistically stable; finer binning can be introduced after checking the
number of daughter samples per bin.

The \(\phi\) coordinate is periodic. The interpolation functions below account
for the \(-\pi/\pi\) boundary.


In [ ]:
# ============================================================
# Configuration
# ============================================================

sample_file = Path(
    "input/map_k0s_pp_pion_qa_v1.root"
)

map_output_file = Path(
    "output/k0s_momentum_scale_map_plots_cut03/k0s_momentum_scale_map_cut03.root"
)

plot_dir = Path(
    "output/k0s_momentum_scale_map_plots_cut03"
)
plot_dir.mkdir(parents=True, exist_ok=True)

sample_tree_name = "momentumScaleTree"

# Calibration binning.
pt_edges = np.array([
    0.20, 0.30, 0.40, 0.55, 0.70,
    0.90, 1.20, 1.50, 2.00, 3.00, 5.00,
], dtype=float)

phi_edges = np.linspace(-math.pi, math.pi, 25)
eta_edges = np.linspace(-1.10, 1.10, 12)

minimum_entries_per_raw_bin = 15
maximum_allowed_mad = 0.12

# Smoothing widths in units of map-bin spacing.
smoothing_sigma_pt_bins = 1.0
smoothing_sigma_phi_bins = 1.2
smoothing_sigma_eta_bins = 1.0

# Reject obviously unstable calibration values.
minimum_scale = 0.75
maximum_scale = 1.25

selected_sides = [0, 1]
selected_charges = [-1, 1]

save_pdf = False
save_png = False


## Read the daughter scale samples

Every row is one \(K^0_S\) daughter. The two daughters from a candidate are
stored separately because they occupy different phase-space coordinates and
may require different corrections.


In [ ]:
# ============================================================
# Read scale samples
# ============================================================

sample_root_file = root.TFile.Open(str(sample_file), "READ")
if not sample_root_file or sample_root_file.IsZombie():
    raise OSError(f"Could not open {sample_file}")

sample_tree = sample_root_file.Get(sample_tree_name)
if not sample_tree:
    raise KeyError(f"Missing tree {sample_tree_name}")

sample_arrays = root.RDataFrame(sample_tree).AsNumpy([
    "side",
    "charge",
    "pt",
    "phi",
    "eta",
    "momentum_scale",
    "mass_before",
    "mass_after",
    "dira_xy_before",
    "dira_xy_after",
])

side_values = np.asarray(sample_arrays["side"], dtype=int)
charge_values = np.asarray(sample_arrays["charge"], dtype=int)
pt_values = np.asarray(sample_arrays["pt"], dtype=float)
phi_values = np.asarray(sample_arrays["phi"], dtype=float)
eta_values = np.asarray(sample_arrays["eta"], dtype=float)
scale_values = np.asarray(
    sample_arrays["momentum_scale"],
    dtype=float,
)

finite_mask = (
    np.isfinite(pt_values)
    & np.isfinite(phi_values)
    & np.isfinite(eta_values)
    & np.isfinite(scale_values)
    & (scale_values >= minimum_scale)
    & (scale_values <= maximum_scale)
)

side_values = side_values[finite_mask]
charge_values = charge_values[finite_mask]
pt_values = pt_values[finite_mask]
phi_values = phi_values[finite_mask]
eta_values = eta_values[finite_mask]
scale_values = scale_values[finite_mask]

print(f"Usable daughter samples: {len(scale_values):,}")
print(
    "Scale median and central 68%:",
    np.median(scale_values),
    np.quantile(scale_values, [0.16, 0.84]),
)


## Robust raw map

For every side and charge, each \((p_T,\phi,\eta)\) bin stores:

- the median momentum scale;
- the median absolute deviation (MAD);
- the number of daughter samples.

The median is used instead of the mean because individual \(K^0_S\) candidates
can still have incorrect pairing or poorly reconstructed tracks.


In [ ]:
# ============================================================
# Build robust raw maps
# ============================================================

pt_centers = 0.5 * (pt_edges[:-1] + pt_edges[1:])
phi_centers = 0.5 * (phi_edges[:-1] + phi_edges[1:])
eta_centers = 0.5 * (eta_edges[:-1] + eta_edges[1:])

shape = (
    len(pt_centers),
    len(phi_centers),
    len(eta_centers),
)

raw_maps = {}

def robust_bin_map(side, charge):
    median_map = np.full(shape, np.nan, dtype=float)
    mad_map = np.full(shape, np.nan, dtype=float)
    count_map = np.zeros(shape, dtype=int)

    selection = (
        (side_values == side)
        & (charge_values == charge)
    )

    selected_pt = pt_values[selection]
    selected_phi = phi_values[selection]
    selected_eta = eta_values[selection]
    selected_scale = scale_values[selection]

    pt_index = np.digitize(selected_pt, pt_edges) - 1
    phi_index = np.digitize(selected_phi, phi_edges) - 1
    eta_index = np.digitize(selected_eta, eta_edges) - 1

    valid = (
        (pt_index >= 0)
        & (pt_index < shape[0])
        & (phi_index >= 0)
        & (phi_index < shape[1])
        & (eta_index >= 0)
        & (eta_index < shape[2])
    )

    pt_index = pt_index[valid]
    phi_index = phi_index[valid]
    eta_index = eta_index[valid]
    selected_scale = selected_scale[valid]

    buckets = {}
    for ipt, iphi, ieta, scale in zip(
        pt_index,
        phi_index,
        eta_index,
        selected_scale,
    ):
        buckets.setdefault(
            (ipt, iphi, ieta),
            [],
        ).append(scale)

    for key, values in buckets.items():
        values = np.asarray(values, dtype=float)
        median = np.median(values)
        mad = 1.4826 * np.median(
            np.abs(values - median)
        )

        median_map[key] = median
        mad_map[key] = mad
        count_map[key] = len(values)

    return {
        "median": median_map,
        "mad": mad_map,
        "count": count_map,
    }

for side in selected_sides:
    for charge in selected_charges:
        raw_maps[(side, charge)] = robust_bin_map(
            side,
            charge,
        )

        number_populated = np.count_nonzero(
            raw_maps[(side, charge)]["count"]
            >= minimum_entries_per_raw_bin
        )

        print(
            f"side={side}, charge={charge:+d}: "
            f"{number_populated} populated raw bins"
        )


## Smooth interpolation

A detector correction should vary smoothly across nearby phase-space bins.
The interpolation therefore uses a Gaussian-weighted average of nearby robust
bin medians.

The weights include:

- the number of entries in the source bin;
- inverse variance from the MAD;
- Gaussian distance in \(p_T\), \(\phi\), and \(\eta\);
- periodic distance in \(\phi\).

This creates a defined correction even in a sparse bin while still giving the
largest weight to nearby, statistically precise calibration bins.


In [ ]:
# ============================================================
# Smooth and fill the maps
# ============================================================

def periodic_phi_bin_distance(first, second, number_of_bins):
    direct = abs(first - second)
    return min(direct, number_of_bins - direct)

def smooth_one_map(raw):
    median_map = raw["median"]
    mad_map = raw["mad"]
    count_map = raw["count"]

    smoothed = np.full(shape, np.nan, dtype=float)
    effective_weight = np.zeros(shape, dtype=float)

    valid_source_bins = np.argwhere(
        (count_map >= minimum_entries_per_raw_bin)
        & np.isfinite(median_map)
        & np.isfinite(mad_map)
        & (mad_map <= maximum_allowed_mad)
    )

    if len(valid_source_bins) == 0:
        return {
            "scale": smoothed,
            "effective_weight": effective_weight,
        }

    for target_pt in range(shape[0]):
        for target_phi in range(shape[1]):
            for target_eta in range(shape[2]):
                weighted_scale = 0.0
                total_weight = 0.0

                for source_pt, source_phi, source_eta in valid_source_bins:
                    delta_pt = (
                        target_pt - source_pt
                    ) / smoothing_sigma_pt_bins

                    delta_phi = periodic_phi_bin_distance(
                        target_phi,
                        source_phi,
                        shape[1],
                    ) / smoothing_sigma_phi_bins

                    delta_eta = (
                        target_eta - source_eta
                    ) / smoothing_sigma_eta_bins

                    distance2 = (
                        delta_pt * delta_pt
                        + delta_phi * delta_phi
                        + delta_eta * delta_eta
                    )

                    geometry_weight = math.exp(
                        -0.5 * distance2
                    )

                    statistical_weight = (
                        count_map[
                            source_pt,
                            source_phi,
                            source_eta,
                        ]
                        / max(
                            mad_map[
                                source_pt,
                                source_phi,
                                source_eta,
                            ] ** 2,
                            1.0e-5,
                        )
                    )

                    weight = (
                        geometry_weight
                        * statistical_weight
                    )

                    weighted_scale += weight * median_map[
                        source_pt,
                        source_phi,
                        source_eta,
                    ]
                    total_weight += weight

                if total_weight > 0:
                    smoothed[
                        target_pt,
                        target_phi,
                        target_eta,
                    ] = weighted_scale / total_weight

                    effective_weight[
                        target_pt,
                        target_phi,
                        target_eta,
                    ] = total_weight

    return {
        "scale": smoothed,
        "effective_weight": effective_weight,
    }

smooth_maps = {}

for key, raw in raw_maps.items():
    smooth_maps[key] = smooth_one_map(raw)

    values = smooth_maps[key]["scale"]
    print(
        f"{key}: smoothed range "
        f"{np.nanmin(values):.5f} to "
        f"{np.nanmax(values):.5f}"
    )


## Continuous map lookup

The lookup below performs trilinear interpolation between neighboring
\(p_T\), \(\phi\), and \(\eta\) bin centers. The \(\phi\) interpolation wraps
periodically across \(-\pi\) and \(+\pi\).

Values outside the calibrated \(p_T\) or \(\eta\) range are clipped to the
nearest map boundary rather than extrapolated.


In [ ]:
# ============================================================
# Continuous trilinear lookup
# ============================================================

def wrap_phi(phi):
    return (phi + math.pi) % (2.0 * math.pi) - math.pi

def interpolation_indices(value, centers):
    value = float(np.clip(value, centers[0], centers[-1]))

    upper = int(np.searchsorted(
        centers,
        value,
        side="right",
    ))

    if upper <= 0:
        return 0, 0, 0.0

    if upper >= len(centers):
        last = len(centers) - 1
        return last, last, 0.0

    lower = upper - 1
    denominator = centers[upper] - centers[lower]
    fraction = (
        0.0
        if denominator <= 0
        else (value - centers[lower]) / denominator
    )

    return lower, upper, fraction

def phi_interpolation_indices(phi):
    phi = wrap_phi(phi)

    extended_centers = np.concatenate([
        phi_centers,
        [phi_centers[0] + 2.0 * math.pi],
    ])

    shifted_phi = phi
    if shifted_phi < phi_centers[0]:
        shifted_phi += 2.0 * math.pi

    upper = int(np.searchsorted(
        extended_centers,
        shifted_phi,
        side="right",
    ))

    lower = max(0, upper - 1)
    upper = min(upper, len(extended_centers) - 1)

    lower_index = lower % len(phi_centers)
    upper_index = upper % len(phi_centers)

    denominator = (
        extended_centers[upper]
        - extended_centers[lower]
    )

    fraction = (
        0.0
        if denominator <= 0
        else (
            shifted_phi
            - extended_centers[lower]
        ) / denominator
    )

    return lower_index, upper_index, fraction

def momentum_scale_lookup(
    side,
    charge,
    pt,
    phi,
    eta,
):
    side = int(side)
    charge = 1 if charge > 0 else -1

    scale_map = smooth_maps[(side, charge)]["scale"]

    ipt0, ipt1, fpt = interpolation_indices(
        pt,
        pt_centers,
    )

    iphi0, iphi1, fphi = phi_interpolation_indices(
        phi
    )

    ieta0, ieta1, feta = interpolation_indices(
        eta,
        eta_centers,
    )

    value = 0.0

    for ipt, weight_pt in [
        (ipt0, 1.0 - fpt),
        (ipt1, fpt),
    ]:
        for iphi, weight_phi in [
            (iphi0, 1.0 - fphi),
            (iphi1, fphi),
        ]:
            for ieta, weight_eta in [
                (ieta0, 1.0 - feta),
                (ieta1, feta),
            ]:
                value += (
                    weight_pt
                    * weight_phi
                    * weight_eta
                    * scale_map[ipt, iphi, ieta]
                )

    return float(value)


## Map visualization

The next plots show \(s(\phi,p_T)\) in selected \(\eta\) slices and
\(s(\phi,\eta)\) in selected \(p_T\) slices for each side and charge.

A useful first check is whether neighboring bins change smoothly without
erasing coherent side-, charge-, or azimuth-dependent structure.


In [ ]:
# ============================================================
# Plot map slices
# ============================================================

keep_objects = []
canvases = []

def keep(obj):
    keep_objects.append(obj)
    return obj

def set_pad():
    root.gPad.SetLeftMargin(0.13)
    root.gPad.SetBottomMargin(0.13)
    root.gPad.SetRightMargin(0.17)
    root.gPad.SetTopMargin(0.07)

def draw_label(text, x=0.14, y=0.94, size=0.030):
    label = keep(root.TLatex())
    label.SetNDC(True)
    label.SetTextFont(42)
    label.SetTextSize(size)
    label.DrawLatex(x, y, text)

def save_canvas(canvas, name):
    canvases.append(canvas)

    if save_pdf:
        canvas.SaveAs(
            str(plot_dir / f"{name}.pdf")
        )

    if save_png:
        canvas.SaveAs(
            str(plot_dir / f"{name}.png")
        )

eta_slice_indices = [
    1,
    len(eta_centers) // 2,
    len(eta_centers) - 2,
]

for side in selected_sides:
    for charge in selected_charges:
        canvas = keep(root.TCanvas(
            f"c_scale_phi_pt_side{side}_q{charge:+d}",
            "",
            1500,
            480,
        ))
        canvas.Divide(3, 1)

        scale_map = smooth_maps[(side, charge)]["scale"]

        for ipad, ieta in enumerate(
            eta_slice_indices,
            start=1,
        ):
            canvas.cd(ipad)
            set_pad()

            hist = keep(root.TH2D(
                (
                    f"h_scale_phi_pt_side{side}_"
                    f"q{charge:+d}_eta{ieta}"
                ),
                "",
                len(phi_edges) - 1,
                phi_edges,
                len(pt_edges) - 1,
                pt_edges,
            ))
            hist.SetDirectory(0)

            for ipt in range(shape[0]):
                for iphi in range(shape[1]):
                    hist.SetBinContent(
                        iphi + 1,
                        ipt + 1,
                        scale_map[ipt, iphi, ieta],
                    )

            hist.GetXaxis().SetTitle("#phi")
            hist.GetYaxis().SetTitle("p_{T} [GeV/c]")
            hist.GetZaxis().SetTitle("momentum scale")
            hist.Draw("COLZ")

            draw_label(
                (
                    f"side {side}, q={charge:+d}, "
                    f"#eta={eta_centers[ieta]:.2f}"
                )
            )

        canvas.Update()
        save_canvas(
            canvas,
            f"scale_phi_pt_side{side}_q{charge:+d}",
        )


## Write the map to ROOT

For each side and charge, the output stores:

- `h3_momentum_scale`: the smoothed scale map;
- `h3_raw_median`: the raw robust-bin median;
- `h3_raw_mad`: the raw-bin MAD;
- `h3_entries`: the number of calibration daughters;
- `h3_effective_weight`: the smoothing support.

The axes are \(p_T\), \(\phi\), and \(\eta\).


In [ ]:
# ============================================================
# Write ROOT map file
# ============================================================

map_output_file.parent.mkdir(
    parents=True,
    exist_ok=True,
)

output = root.TFile.Open(
    str(map_output_file),
    "RECREATE",
)

if not output or output.IsZombie():
    raise OSError(
        f"Could not create {map_output_file}"
    )

root.TNamed(
    "map_definition",
    (
        "momentum scale as a function of "
        "side, charge, pt, phi, eta; "
        "raw median plus Gaussian neighbor smoothing"
    ),
).Write()

for side in selected_sides:
    side_directory = output.mkdir(f"side{side}")

    for charge in selected_charges:
        charge_name = (
            "qplus"
            if charge > 0
            else "qminus"
        )
        directory = side_directory.mkdir(charge_name)
        directory.cd()

        raw = raw_maps[(side, charge)]
        smooth = smooth_maps[(side, charge)]

        histograms = {
            "h3_momentum_scale": smooth["scale"],
            "h3_raw_median": raw["median"],
            "h3_raw_mad": raw["mad"],
            "h3_entries": raw["count"],
            "h3_effective_weight": smooth[
                "effective_weight"
            ],
        }

        for name, values in histograms.items():
            hist = root.TH3D(
                name,
                (
                    f"{name};p_{{T}} [GeV/c];"
                    "#phi;#eta"
                ),
                len(pt_edges) - 1,
                pt_edges,
                len(phi_edges) - 1,
                phi_edges,
                len(eta_edges) - 1,
                eta_edges,
            )
            hist.SetDirectory(directory)

            for ipt in range(shape[0]):
                for iphi in range(shape[1]):
                    for ieta in range(shape[2]):
                        value = values[
                            ipt,
                            iphi,
                            ieta,
                        ]

                        if np.isfinite(value):
                            hist.SetBinContent(
                                ipt + 1,
                                iphi + 1,
                                ieta + 1,
                                float(value),
                            )

            hist.Write()

output.Close()
print(f"Wrote {map_output_file}")


# Optional validation on pair candidates

The following section applies the map to pair momenta and recomputes masses for:

- \(K^0_S\to\pi^+\pi^-\);
- \(\Lambda\to p\pi^-\);
- \(\bar\Lambda\to\bar p\pi^+\);
- \(\phi\to K^+K^-\);
- \(D^0\to\pi^+K^-\);
- \(\bar D^0\to K^+\pi^-\).

The validation should preferably use files or events not used to construct the
map. Otherwise the improvement in the \(K^0_S\) mass is partly a closure test.

The pair tree may not store a TPC side for every non-\(K^0_S\) candidate.
When a side branch is unavailable, the fallback used here is:

\[
\mathrm{side}=1\ \text{for}\ \eta\ge0,\qquad
\mathrm{side}=0\ \text{for}\ \eta<0.
\]

That approximation must be replaced by the actual track side when such a
branch is added to the pair tree.


In [ ]:
# ============================================================
# Validation configuration
# ============================================================

run_validation = False

validation_input_pattern = (
    "/path/to/independent/p_v0_*.root"
)
validation_tree_name = "pairTree"
maximum_validation_entries = 2_000_000

particle_masses = {
    "pi": 0.13957039,
    "K": 0.493677,
    "p": 0.9382720813,
}

validation_channels = {
    "KShort": {
        "mask": 1,
        "mass1": particle_masses["pi"],
        "mass2": particle_masses["pi"],
        "range": (0.40, 0.60),
        "bins": 240,
    },
    "Lambda": {
        "mask": 2,
        "mass1": particle_masses["p"],
        "mass2": particle_masses["pi"],
        "range": (1.06, 1.18),
        "bins": 240,
    },
    "AntiLambda": {
        "mask": 4,
        "mass1": particle_masses["pi"],
        "mass2": particle_masses["p"],
        "range": (1.06, 1.18),
        "bins": 240,
    },
    "Phi": {
        "mask": 8,
        "mass1": particle_masses["K"],
        "mass2": particle_masses["K"],
        "range": (0.98, 1.08),
        "bins": 240,
    },
    "D0": {
        "mask": 16,
        "mass1": particle_masses["pi"],
        "mass2": particle_masses["K"],
        "range": (1.70, 2.02),
        "bins": 240,
    },
    "AntiD0": {
        "mask": 32,
        "mass1": particle_masses["K"],
        "mass2": particle_masses["pi"],
        "range": (1.70, 2.02),
        "bins": 240,
    },
}


In [ ]:
# ============================================================
# Validation helpers
# ============================================================

def eta_from_components(px, py, pz):
    momentum = math.sqrt(
        px * px + py * py + pz * pz
    )

    numerator = momentum + pz
    denominator = momentum - pz

    if numerator <= 0 or denominator <= 0:
        return float("nan")

    return 0.5 * math.log(
        numerator / denominator
    )

def invariant_mass_components(
    px1,
    py1,
    pz1,
    mass1,
    px2,
    py2,
    pz2,
    mass2,
):
    p1_squared = (
        px1 * px1
        + py1 * py1
        + pz1 * pz1
    )
    p2_squared = (
        px2 * px2
        + py2 * py2
        + pz2 * pz2
    )

    energy1 = math.sqrt(
        p1_squared + mass1 * mass1
    )
    energy2 = math.sqrt(
        p2_squared + mass2 * mass2
    )

    total_px = px1 + px2
    total_py = py1 + py2
    total_pz = pz1 + pz2

    mass_squared = (
        (energy1 + energy2) ** 2
        - total_px ** 2
        - total_py ** 2
        - total_pz ** 2
    )

    return math.sqrt(max(0.0, mass_squared))

def inferred_side_from_eta(eta):
    return 1 if eta >= 0 else 0


In [ ]:
# ============================================================
# Apply the map and compare masses
# ============================================================

validation_histograms = {}

if run_validation:
    validation_chain = root.TChain(
        validation_tree_name
    )
    number_of_files = validation_chain.Add(
        validation_input_pattern
    )

    if number_of_files <= 0:
        raise OSError(
            "No validation files matched "
            f"{validation_input_pattern}"
        )

    validation_frame = root.RDataFrame(
        validation_chain
    )

    if maximum_validation_entries > 0:
        validation_frame = validation_frame.Range(
            maximum_validation_entries
        )

    branches = [
        "candidate_mask",
        "charge1",
        "charge2",
        "px1",
        "py1",
        "pz1",
        "px2",
        "py2",
        "pz2",
    ]

    arrays = validation_frame.AsNumpy(branches)

    number_of_rows = len(arrays["candidate_mask"])
    print(f"Validation rows: {number_of_rows:,}")

    for channel, configuration in validation_channels.items():
        low, high = configuration["range"]

        before = keep(root.TH1D(
            f"h_{channel}_mass_before_map",
            (
                f"{channel};mass [GeV/c^{{2}}];"
                "candidates"
            ),
            configuration["bins"],
            low,
            high,
        ))
        before.SetDirectory(0)

        after = keep(root.TH1D(
            f"h_{channel}_mass_after_map",
            (
                f"{channel};mass [GeV/c^{{2}}];"
                "candidates"
            ),
            configuration["bins"],
            low,
            high,
        ))
        after.SetDirectory(0)

        validation_histograms[channel] = {
            "before": before,
            "after": after,
        }

    for index in range(number_of_rows):
        candidate_mask = int(
            arrays["candidate_mask"][index]
        )

        charge1 = float(arrays["charge1"][index])
        charge2 = float(arrays["charge2"][index])

        px1 = float(arrays["px1"][index])
        py1 = float(arrays["py1"][index])
        pz1 = float(arrays["pz1"][index])
        px2 = float(arrays["px2"][index])
        py2 = float(arrays["py2"][index])
        pz2 = float(arrays["pz2"][index])

        pt1 = math.hypot(px1, py1)
        pt2 = math.hypot(px2, py2)

        phi1 = math.atan2(py1, px1)
        phi2 = math.atan2(py2, px2)

        eta1 = eta_from_components(
            px1,
            py1,
            pz1,
        )
        eta2 = eta_from_components(
            px2,
            py2,
            pz2,
        )

        if not (
            math.isfinite(eta1)
            and math.isfinite(eta2)
        ):
            continue

        side1 = inferred_side_from_eta(eta1)
        side2 = inferred_side_from_eta(eta2)

        scale1 = momentum_scale_lookup(
            side1,
            charge1,
            pt1,
            phi1,
            eta1,
        )
        scale2 = momentum_scale_lookup(
            side2,
            charge2,
            pt2,
            phi2,
            eta2,
        )

        corrected_px1 = scale1 * px1
        corrected_py1 = scale1 * py1
        corrected_pz1 = scale1 * pz1

        corrected_px2 = scale2 * px2
        corrected_py2 = scale2 * py2
        corrected_pz2 = scale2 * pz2

        for channel, configuration in validation_channels.items():
            if (
                candidate_mask
                & configuration["mask"]
            ) == 0:
                continue

            mass_before = invariant_mass_components(
                px1,
                py1,
                pz1,
                configuration["mass1"],
                px2,
                py2,
                pz2,
                configuration["mass2"],
            )

            mass_after = invariant_mass_components(
                corrected_px1,
                corrected_py1,
                corrected_pz1,
                configuration["mass1"],
                corrected_px2,
                corrected_py2,
                corrected_pz2,
                configuration["mass2"],
            )

            validation_histograms[channel][
                "before"
            ].Fill(mass_before)

            validation_histograms[channel][
                "after"
            ].Fill(mass_after)

    print("Validation histograms filled")
else:
    print(
        "run_validation = False; "
        "set it to True after configuring "
        "an independent pair-tree sample."
    )


## Validation interpretation

A useful correction should behave consistently across channels:

- the peak position should move toward the expected mass;
- the width should decrease or at least not grow significantly;
- no strong nonphysical structure should be introduced;
- positive and negative particle channels should respond consistently;
- improvements should remain when the calibration and validation samples are separated.

A correction that improves only the calibration \(K^0_S\) sample but worsens
\(\Lambda\), \(\phi\), or \(D^0\) is likely overfit or based on an incorrect
phase-space dependence.


In [ ]:
# ============================================================
# Draw validation masses
# ============================================================

if run_validation:
    channel_names = list(
        validation_channels.keys()
    )

    c_validation = keep(root.TCanvas(
        "c_validation_masses",
        "c_validation_masses",
        1600,
        1000,
    ))
    c_validation.Divide(3, 2)

    for ipad, channel in enumerate(
        channel_names,
        start=1,
    ):
        c_validation.cd(ipad)
        root.gPad.SetLeftMargin(0.13)
        root.gPad.SetBottomMargin(0.13)
        root.gPad.SetRightMargin(0.04)
        root.gPad.SetTopMargin(0.07)

        before = validation_histograms[
            channel
        ]["before"]
        after = validation_histograms[
            channel
        ]["after"]

        before.SetLineWidth(3)
        after.SetLineWidth(3)
        after.SetLineStyle(2)

        maximum = max(
            before.GetMaximum(),
            after.GetMaximum(),
        )
        before.SetMaximum(1.18 * maximum)

        before.Draw("HIST")
        after.Draw("HIST SAME")

        legend = keep(root.TLegend(
            0.60,
            0.75,
            0.89,
            0.89,
        ))
        legend.SetBorderSize(0)
        legend.SetFillStyle(0)
        legend.AddEntry(
            before,
            "before map",
            "l",
        )
        legend.AddEntry(
            after,
            "after map",
            "l",
        )
        legend.Draw()

        draw_label(channel)

    c_validation.Update()
    save_canvas(
        c_validation,
        "mass_validation_all_channels",
    )
    c_validation
